In [8]:
import os

import pandas as pd
import plotly.express as px
import streamlit as st

from hw2_labeler import label_comment

# ---- Settings ----------------------------------------------------------

TITLE = "Patient Voice: Feedback Analytics"
LOW_RATING, HIGH_RATING = 2, 4
CSV_FILE = "feedback.csv"
COLORS = {
    "Scheduling": "#5379B5",
    "Wait Times": "#D98C45",
    "Staff Experience": "#8E6BB5",
    "Positive Experience": "#2BB38A",
    "Needs Review": "#B34700"
}

# ------------------------------------------------------------------------

st.set_page_config(page_title=TITLE, page_icon="🩺", layout="wide")
st.title(TITLE)
st.caption("Share your healthcare experience. Patient feedback helps identify opportunities to improve service quality.")

# Load the saved feedback, or start with an empty table
if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
else:
    df = pd.DataFrame(columns=["Comment", "Rating", "Label"])

tab_customer, tab_owner = st.tabs(["Patient: Leave Feedback", "Management: Analyze Feedback"])

with tab_customer:
    st.subheader("Tell Us About Your Experience")

    with st.form("feedback_form", clear_on_submit=True):
        comment = st.text_area("Share Your Feedback (optional)")
        stars = st.feedback("Stars")
        submitted = st.form_submit_button("Submit Feedback")

    if submitted:
        if not comment.strip() and stars is None:
            st.error("Please provide a comment, a star rating, or both.")
        else:
            if comment.strip():
                label = label_comment(comment.strip())
            else:
                label = "No Comment"

            new_row = pd.DataFrame([{
                "comment": comment.strip(),
                "rating": stars + 1 if stars is not None else None,
                "label": label
            }])

            pd.concat([df, new_row], ignore_index=True).to_csv(CSV_FILE, index=False)
            df = pd.read_csv(CSV_FILE)

            st.success("Thank you for sharing your feedback!")

            if comment.strip():
                st.caption(f"AI Classification: {label}")

with tab_owner:
    if df.empty:
        st.info("No feedback yet. Switch to the patient tab to submit the first review.")
    else:
        # Identify comments that require management review
        flag = (df["label"] == "Needs Review") | (
            (df["rating"] <= LOW_RATING) &
            (df["comment"].fillna("").str.strip() != "")
        )

        st.subheader("Patient Experience Overview")
        m1, m2, m3 = st.columns(3)
        m1.metric("Total Feedback", len(df))
        m2.metric(
            "Average Rating",
            f"{df['rating'].mean():.1f}" if df["rating"].notna().any() else "N/A"
        )
        m3.metric("Needs Manual Review", int(flag.sum()))

        col1, col2 = st.columns(2)

        with col1:
            st.caption("Patient Feedback by Category")

            counts = df[df["label"] != "No Comment"]["label"].value_counts().rename_axis("label").reset_index(name="count")

            if not counts.empty:
                fig = px.bar(
                    counts,
                    x="label",
                    y="count",
                    color="label",
                    text="count",
                    color_discrete_map=COLORS
                )

                fig.update_layout(
                    showlegend=False,
                    xaxis_title=None,
                    yaxis_title="Number of Comments"
                )

                st.plotly_chart(fig, use_container_width=True)
            else:
                st.info("No written feedback available yet.")

        st.subheader("Feedback Requiring Management Review")

        st.caption(
            "Comments are flagged when the AI identifies a concern requiring review "
            "or when a patient provides a rating of 2 stars or lower with a written comment."
        )

        st.dataframe(df[flag], hide_index=True, width="stretch")

        with st.expander("All feedback"):

          st.dataframe(df, hide_index=True, width="stretch")

        st.download_button(
            "Download CSV",
            df.to_csv(index=False),
            file_name="feedback.csv"
        )

        st.caption(
            "On the free Streamlit Cloud, the saved file can disappear when the app restarts. "
            "Download your CSV to keep it."
        )



ModuleNotFoundError: No module named 'streamlit'